In [1]:
from typing import Any
from collections import Counter

import pandas as pd
from collections import Counter
from sklearn.datasets import load_iris
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [2]:
# Import nested_cv.py
from nested_cv import evaluate_nested_cv

In [3]:
# 1. Dataset: Binary Classification (Versicolor vs Virginica)
X, y = load_iris(return_X_y=True)
X = X[50:150]
y = y[50:150]

In [4]:
# 2. Define a generic Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('model', RandomForestClassifier()) 
])

In [5]:
# 3. Define a Multi-Estimator Grid (Random Forest vs SVM)
param_grid = [
    {
        'model': [RandomForestClassifier(random_state=42)],
        'model__n_estimators': [50, 100],
        'model__max_depth': [5, None],
        'model__min_samples_split': [2, 5]
    },
    {
        'model': [SVC(random_state=42)],
        'model__C': [0.1, 1.0, 10.0],
        'model__kernel': ['linear', 'rbf']
    }
]

In [6]:
# 4. Standard Binary Classification Metrics
metrics = ['accuracy', 'precision', 'recall', 'f1']

In [7]:
# 5. Execute Multi-Model Nested CV
print("Starting Advanced Binary Nested CV (RandomForest vs SVM)...\n")
results = evaluate_nested_cv(
    X=X,
    y=y,
    estimator=pipeline,
    param_grid=param_grid,
    metrics=metrics,
    refit_metric='f1',  # Optimize for F1-Score
    outer_splits=5,
    inner_splits=3,
    is_classification=True,
    random_state=42,
    verbose=False  
)

Starting Advanced Binary Nested CV (RandomForest vs SVM)...



In [9]:
print("=== 0. DICTIONARY KEYS EXPOSURE ===")
print(f"Available keys: {list(results.keys())}")
print(f"Configuration Used: {results['config']}")

=== 0. DICTIONARY KEYS EXPOSURE ===
Available keys: ['metrics', 'model_ranking', 'best_params_per_fold', 'fold_details', 'best_model', 'config']
Configuration Used: {'outer_splits': 5, 'inner_splits': 3, 'refit_metric': 'f1'}


In [10]:
print("\n=== 1. OVERALL UNBIASED PERFORMANCE (Generalization Error) ===")
for metric, stats in results['metrics'].items():
    print(f"{metric.capitalize()}: {stats['mean']} (±{stats['std']})")


=== 1. OVERALL UNBIASED PERFORMANCE (Generalization Error) ===
Accuracy: 0.93 (±0.06)
Precision: 0.9303 (±0.0855)
Recall: 0.94 (±0.08)
F1: 0.931 (±0.0587)


In [11]:
print("\n=== 2. DETAILED PER-FOLD ANALYSIS ===")
for fold in results['fold_details']:
    print(f"Fold {fold['fold']}:")
    print(f"  -> Best Inner Params: {fold['best_params']}")
    print(f"  -> Outer Test Scores: {fold['scores']}")


=== 2. DETAILED PER-FOLD ANALYSIS ===
Fold 1:
  -> Best Inner Params: {'model': SVC(C=10.0, random_state=42), 'model__C': 10.0, 'model__kernel': 'linear'}
  -> Outer Test Scores: {'accuracy': 0.9, 'precision': 1.0, 'recall': 0.8, 'f1': 0.8889}
Fold 2:
  -> Best Inner Params: {'model': SVC(C=10.0, random_state=42), 'model__C': 0.1, 'model__kernel': 'linear'}
  -> Outer Test Scores: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
Fold 3:
  -> Best Inner Params: {'model': SVC(C=10.0, random_state=42), 'model__C': 0.1, 'model__kernel': 'rbf'}
  -> Outer Test Scores: {'accuracy': 0.85, 'precision': 0.8182, 'recall': 0.9, 'f1': 0.8571}
Fold 4:
  -> Best Inner Params: {'model': SVC(C=10.0, random_state=42), 'model__C': 1.0, 'model__kernel': 'rbf'}
  -> Outer Test Scores: {'accuracy': 0.9, 'precision': 0.8333, 'recall': 1.0, 'f1': 0.9091}
Fold 5:
  -> Best Inner Params: {'model': SVC(C=10.0, random_state=42), 'model__C': 1.0, 'model__kernel': 'linear'}
  -> Outer Test Scores: {'

In [12]:
print("\n=== 3. HYPERPARAMETER STABILITY ===")
param_strings = [str(p) for p in results['best_params_per_fold']]
most_common_params, count = Counter(param_strings).most_common(1)[0]
total_folds = results['config']['outer_splits']
print(f"Most frequent inner loop configuration ({count}/{total_folds} times):")
print(most_common_params)


=== 3. HYPERPARAMETER STABILITY ===
Most frequent inner loop configuration (1/5 times):
{'model': SVC(C=10.0, random_state=42), 'model__C': 10.0, 'model__kernel': 'linear'}


In [13]:
print("\n=== 4. GLOBAL MODEL RANKING (Top 5 Configurations) ===")
for i, model_info in enumerate(results['model_ranking'][:5], 1):
    m_name = model_info['model']
    m_score = model_info['mean_outer_score']
    m_std = model_info['std_outer_score']
    m_params = str(model_info['params'])
        
    if len(m_params) > 60:
        m_params = m_params[:57] + "..."
            
    print(f"{i}. {m_name:<15} | Score: {m_score:.4f} (±{m_std:.4f}) | Params: {m_params}")


=== 4. GLOBAL MODEL RANKING (Top 5 Configurations) ===
1. SVC             | Score: 0.9396 (±0.0497) | Params: {'model': SVC(C=10.0, random_state=42), 'model__C': 10.0,...
2. SVC             | Score: 0.9357 (±0.0622) | Params: {'model': SVC(C=10.0, random_state=42), 'model__C': 0.1, ...
3. RandomForestClassifier | Score: 0.9310 (±0.0587) | Params: {'model': RandomForestClassifier(min_samples_split=5, ran...
4. RandomForestClassifier | Score: 0.9310 (±0.0587) | Params: {'model': RandomForestClassifier(min_samples_split=5, ran...
5. RandomForestClassifier | Score: 0.9310 (±0.0587) | Params: {'model': RandomForestClassifier(min_samples_split=5, ran...


In [14]:
print("\n=== 5. BEST MODEL FOR DEPLOYMENT ===")
final_model = results['best_model']
print(f"Algorithm  : {final_model.__class__.__name__}")
print(f"Parameters : {final_model.get_params()}")


=== 5. BEST MODEL FOR DEPLOYMENT ===
Algorithm  : Pipeline
Parameters : {'memory': None, 'steps': [('scaler', StandardScaler()), ('model', SVC(C=10.0, random_state=42))], 'verbose': False, 'scaler': StandardScaler(), 'model': SVC(C=10.0, random_state=42), 'scaler__copy': True, 'scaler__with_mean': True, 'scaler__with_std': True, 'model__C': 10.0, 'model__break_ties': False, 'model__cache_size': 200, 'model__class_weight': None, 'model__coef0': 0.0, 'model__decision_function_shape': 'ovr', 'model__degree': 3, 'model__gamma': 'scale', 'model__kernel': 'rbf', 'model__max_iter': -1, 'model__probability': False, 'model__random_state': 42, 'model__shrinking': True, 'model__tol': 0.001, 'model__verbose': False}
